# Quick Demo: Enterprise AI Deployment Preference Analysis

Minimal notebook to load processed data and run a quick model evaluation.

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

# Load Stage 1 panel
df = pd.read_csv('../data/processed/eurostat_ai_panel.csv')
print(f'Stage 1 panel: {df.shape}')
print(f'Geo groups: {df["geo"].nunique()}')
print(f'Years: {sorted(df["time_period"].unique())}')

In [ ]:
# Quick Ridge evaluation with GroupKFold
target = 'target_workflow_automation'
exclude = [target, 'geo', 'time_period']
features = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64', 'int64']]

X = df[features].fillna(0)
y = df[target]
groups = df['geo']

gkf = GroupKFold(n_splits=5)
scores = []
for train_idx, test_idx in gkf.split(X, y, groups):
    model = Ridge(alpha=1.0)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    scores.append(r2_score(y.iloc[test_idx], pred))

print(f'GroupKFold R2: {pd.Series(scores).mean():.4f} (+/- {pd.Series(scores).std():.4f})')